# Hands-on with `CUDA.jl` and friends

In this notebook, we'll use [`CUDA.jl`](https://github.com/JuliaGPU/CUDA.jl) to control and program NVIDIA GPU resources. If you have an NVIDIA GPU already (in your laptop or desktop), you can run this notebook locally. If not, you can use a GPU available through [Google Colab](https://colab.research.google.com/). This will allow you to use a GPU *for free* for up to 12 hours of interactive experimentation.

We'll use `CUDA.jl` because:
- NVIDIA GPUs are highly available
- CUDA is (for now) the most mature GPU programming paradigm with many libraries and resources
- Many of the underlying concepts transfer to other systems like AMD ROCm, Apple Metal, and Intel oneAPI

Let's import `CUDA.jl`:

In [1]:
using Pkg; Pkg.add("CUDA"); Pkg.add("StaticArrays")
using CUDA, StaticArrays

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.12/Project.toml`
  [90137ffa] + StaticArrays v1.9.18
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


CUDA is actually made up of several components:
- The GPU driver, which controls the graphics card and its interactions with the operating system kernel
- The CUDA runtime, which provides the basic CUDA C extensions, CUDA API, and CUDA compiler
- The CUDA libraries, which provide a "GPUified" implementation of BLAS (CUBLAS), RNG (CURAND), and many other things

`CUDA.jl` has prepackaged [artifacts](https://pkgdocs.julialang.org/v1/artifacts/) for many CUDA versions, Julia versions, and OSes. It will download the appropriate one by default and install it for you. This lets us use the high-level Julia functions for CUDA arrays without worrying about the backend API calls (for now). Once `CUDA.jl` has installed, we can try creating some arrays on the GPU. We do this with `CuArray`, which turns a CPU array into a CUDA array by copying it.

`CUDA.jl` also has some convenience constructors for `CuArray`s, like `CUDA.zeros`, `CUDA.rand`, and `undef` initializers. We can even use methods like `map` and `reduce` on these.

In [13]:
cu_a = CUDA.rand(Float32, 2, 2);
println(cu_a)

Float32[0.9097028 0.3688886; 0.3771052 0.10440259]


In [14]:
cu_a = CUDA.rand(Float64, 1024, 1024);
typeof(cu_a)

CuArray{Float64, 2, CUDACore.DeviceMemory}

In [16]:
cu_b = map(x->x^2, cu_a);

These `CuArray`s live in **device memory** (GPU RAM), which means their individual elements are only available **from code executing on the GPU**. We know that the GPU memory is relatively limited compared to CPU memory, so how does `CUDA.jl` handle this? Do we have to explicitly "reap" these arrays ourselves?

`CUDA.jl` extends the Julia garbage collector to work with GPU arrays as well. It manages a memory pool of GPU memory and garbage-collected `CuArray`s are in fact returned to the memory pool (rather than being entirely deallocated) in order to reduce GC pressure. `CUDA.jl` recycles arrays in the pool preferentially to speed up future allocations and keep an eye on how close we are to running out of memory. We can inspect the current state of the memory pool with `CUDA.pool_status()`:

In [17]:
CUDA.pool_status()

Effective GPU memory usage: 1.16% (172.875 MiB/14.563 GiB)
Memory pool usage: 32.049 MiB (64.000 MiB reserved)


Let's allocate another device array and see how the pool status changes:

In [19]:
cu_c = CUDA.rand(Float64, 1024, 1024)
cu_c .+= cu_c' # make cu_c symmetric
CUDA.pool_status()

Effective GPU memory usage: 1.16% (172.875 MiB/14.563 GiB)
Memory pool usage: 56.049 MiB (64.000 MiB reserved)


We should never need to explicitly deallocate CUDA arrays, but we can reduce pressure on the garbage collector and help `CUDA.jl` out by indicating when we are done with an array, using `CUDA.unsafe_free!`. As you can guess from the name, you should **never** refer to the array after you've unsafely freed it!

In [22]:
CUDA.unsafe_free!(cu_b)
CUDA.pool_status()

Effective GPU memory usage: 1.16% (172.875 MiB/14.563 GiB)
Memory pool usage: 48.049 MiB (64.000 MiB reserved)


### Tips and tricks for higher level array operations

In general, a `map` or `broadcast` over columns/slices launches multiple independent kernels which is usually slower than launching one larger kernel. For example,
```julia
broadcast(eachcol(a)) do x
    prod(x)
end
```
will usually be slower than

```julia
prod(a; dims=2)
```

`CUDA.jl` isn't a tensor compiler and generally implements scalar (single-element) kernels for many operations like `map`. It's up to you to effectively use tools like broadcasting to avoid this.

Broadcast fusion is especially helpful for GPU arrays because it turns multiple kernel launches on the same data into one.

## Using `LinearAlgebra` extensions

Already we can see many of the nice array features we covered yesterday apply just as well to `CuArray`s. Many, but not all. `view`s, for example, are inconsistently supported. We can also make use of many wrapped linear algebra functions optimized for GPU via `CUBLAS`. We'll set the `CUBLAS` logger to prove to ourselves that it is in fact being used from our nice compact call:

In [25]:
using LinearAlgebra
CUBLAS.cublasLoggerConfigure(1, 0, 1, C_NULL) # normally we don't do this, just for demonstration purposes here
cu_d = cu_c * cu_a;

I! cuBLAS (v12.8.4) function cublasStatus_t cublasCreate_v2(cublasContext**) called:
i!  handle: type=cublasHandle_t; val=POINTER (IN HEX:0x0x7f902a4858d0)
i! Time: 2026-06-25T00:07:14 elapsed from start 21.316667 minutes or 1279.000000 seconds
i!Process=1479; Thread=140259119231168; GPU=0; Handle=POINTER (IN HEX:0x(nil))
i! COMPILED WITH: GNU GCC/G++ / 8.5.0 20210514 (Red Hat 8.5.0-22)
I! cuBLAS (v12.8.4) function cublasStatus_t cublasSetStream_v2(cublasHandle_t, cudaStream_t) called:
i!  handle: type=cublasHandle_t; val=POINTER (IN HEX:0x0x1ab7d500)
i!  streamId: type=SOME TYPE; val=POINTER (IN HEX:0x0x18168930)
i! Time: 2026-06-25T00:07:14 elapsed from start 21.316667 minutes or 1279.000000 seconds
i!Process=1479; Thread=140259119231168; GPU=0; Handle=POINTER (IN HEX:0x0x1ab7d500); StreamId=POINTER (IN HEX:0x(nil)) (defaultStream); MathMode=CUBLAS_DEFAULT_MATH
i! COMPILED WITH: GNU GCC/G++ / 8.5.0 20210514 (Red Hat 8.5.0-22)
I! cuBLAS (v12.8.4) function cublasStatus_t cublasSetPoint

We also have wrappers for some factorizations:

In [26]:
qr_c = qr(CUDA.rand(Float64, 64, 32));

I! cuBLAS (v12.8.4) function cublasStatus_t cublasCreate_v2(cublasContext**) called:
i!  handle: type=cublasHandle_t; val=POINTER (IN HEX:0x0x1af5e090)
i! Time: 2026-06-25T00:07:40 elapsed from start 21.750000 minutes or 1305.000000 seconds
i!Process=1479; Thread=140259119231168; GPU=0; Handle=POINTER (IN HEX:0x(nil))
i! COMPILED WITH: GNU GCC/G++ / 8.5.0 20210514 (Red Hat 8.5.0-22)
I! cuBLAS (v12.8.4) function cublasStatus_t cublasCreate_v2(cublasContext**) called:
i!  handle: type=cublasHandle_t; val=POINTER (IN HEX:0x0x1af5e0d8)
i! Time: 2026-06-25T00:07:40 elapsed from start 21.750000 minutes or 1305.000000 seconds
i!Process=1479; Thread=140259119231168; GPU=0; Handle=POINTER (IN HEX:0x(nil))
i! COMPILED WITH: GNU GCC/G++ / 8.5.0 20210514 (Red Hat 8.5.0-22)
I! cuBLAS (v12.8.4) function cublasStatus_t cublasCreate_v2(cublasContext**) called:
i!  handle: type=cublasHandle_t; val=POINTER (IN HEX:0x0x1af5e0e0)
i! Time: 2026-06-25T00:07:40 elapsed from start 21.750000 minutes or 1305.00

## Running multiple functions simultaneously

A `1024 x 1024` matrix is decently hefty but not big enough to saturate the throughput of the GPU. If we have many such matrices, we can spawn multiple separate simultaneous executions on the same GPU to speed up our computation, provided we don't run out of memory. `CUDA.jl` has a nice integration with the built-in Julia threading for this.

In [28]:
n_arrs = 100
cpu_arrays = [rand(Float64, 1024, 1024) for ix_arr in 1:n_arrs]
results = Vector{Float64}(undef, n_arrs)
@sync begin
    for ix_arr in 1:n_arrs
        Threads.@spawn begin
            results[ix_arr] = mapreduce(sin, *, CuArray(cpu_arrays[ix_arr]))
        end
    end
end

Obviously this example is a bit contrived, but using this basic pattern allows Julia and CUDA to work together to interleave memory copies and useful work, making better overall use of your GPU. We wrap everything in the `@sync` block because `Threads.@spawn` returns immediately after creating and launching its task, and `@sync` forces Julia to wait for all the `@spawn`-ed tasks to complete (and thus for all elements of `results` to be populated).

For linear algebra operations, `cuBLAS` also provides *batched* versions of common functions like `gemm` (matrix multiplication) or `matinv` (matrix inverse). These perform the operation in question on multiple "sets" of inputs simulataneously, and can be *substantially* more performant than launching multiple independent kernels.

## Important considerations for memory transfers

- Memory copies to/from/across the GPU are **high latency**. Each one takes a relatively long time to begin. Thus, it's usually better to do **one large copy** (by concatenating arrays, for example) than **many small copies**.
- By default, `CuArray`s live in the GPU memory and can't be accessed elementwise from the CPU. You, as the programmer, explicitly request memory copies, which helps you control and understand when they happen. However, you can use the CUDA [Unified Memory](https://developer.nvidia.com/blog/unified-memory-cuda-beginners/) [from `CUDA.jl`](https://cuda.juliagpu.org/dev/usage/memory/#Unified-memory), which makes the arrays visible from both CPU and GPU. The CUDA driver copies memory back and forth for you as needed. This can be very convenient for the programmer but can lead to degraded performance depending on your memory access pattern.
- If you have a pre-allocated CPU array you'll be copying to and from frequently (a pre-allocated out of loop buffer, for example), you can **pin** the CPU memory using `CUDA.pin`, which can speed up these copies substantially. The very act of pinning is itself time-consuming, so be sure it's worth it (benchmarks!).

## When the builtin wrappers aren't enough: writing our own kernels 🌽

Sometimes the high level Julia constructs just aren't enough. We need to implement our own high-performance functions that will be executed across the massive thread population. Awesome! However, there are some new complications:
- We have to make efficient use of the hardware, which isn't always intuitive
- We have a restricted subset of the programming language available on-GPU
- For many operations it's not obvious how to translate the algorithm to an efficient GPU implementation

A good example of this latter point is creating an [alias table](https://github.com/LilithHafner/AliasTables.jl/) for discrete categorical sampling (this is a common thing to do for sampling state vectors, for example). Many CPU algorithms exist for this purpose, and some GPU ones too, but the underlying approach is quite different because of the GPU's constraints.

A function we will execute in a massively/"embarrassingly" parallel way on the GPU across a large array is called a *kernel*. We can write a restricted subset of Julia within a kernel. `CUDA.jl` and its backend `GPUCompiler.jl` handle translating & compiling this through LLVM to the underlying GPU IR.

The higher-level functions we worked with before are actually implemented behind-the-scenes as CUDA kernels. It's worth spending a little time to understand what is happening when a kernel executes.



### Quick guide to kernel launches and execution

CUDA spawns a *grid* of *blocks* of *threads*, and each thread is an individual worker. Threads are grouped into *warps* of size 32. Each thread in the warp steps through the function instructions **together**. If two threads in the same warp encounter **different** instructions, the warp can no longer execute in parallel and the performance benefit of using the GPU can disappear. This is called **warp divergence** (very *Star Trek*). In general, one warp out of 10,000 experiencing divergence will not hurt things too badly -- but a large fraction of warps experiencing it will.

**Our goal when writing CUDA kernels is to make efficient use of warp-parallelism.**

When we launch the CUDA kernel, we launch it with `(n_blocks_x, n_blocks_y, n_blocks_z)` blocks of `(n_threads_x, n_threads_y, n_threads_z)` threads. Any of these numbers can be 1 -- but you can also create 3D grids if it's more natural for your problem. In general, it's **not** the case that `n_blocks_x * n_blocks_y * n_blocks_z * n_threads_x * n_threads_y * n_threads_z` threads execute in parallel (though this can happen if the sizes are small). Rather, blocks are scheduled by the CUDA driver, so that block 0 may execute, and then block 65, then block 32. **Block execution order is not guaranteed!**

Because the warps are size 32, it is most efficient to make the thread dimensions a power of 2. So, `n_threads_x = 64` would be a good choice, or `n_threads_y = 512`. **In general**, larger thread counts are better (512 threads-per-block is usually more performant than 64) but of course there are some edge cases.

Within a kernel, each thread has access to some information about its personal location in the overall grid. We can query the `threadIdx()` and `blockIdx()` functions to get the thread's `x`, `y`, and `z` thread and block positions, and `blockDim()` to get the number of threads in a block in the `x`, `y`, and `z` directions.

We start by writing a `function` which may take arguments. `CUDA.jl` kernels aren't yet hooked up to standard Julia IO, but we **can** do some basic printing using the `@cuprintln` macro. **CUDA kernels must always return `nothing`.**

In [29]:
function my_first_cuda_kernel(a)
    t_x = threadIdx().x
    b_x = blockIdx().x
    l_a = length(a)
    @cuprintln("Hi! My thread index is $t_x and my block index is $b_x. The length of my argument is $l_a.")
    return
end

my_first_cuda_kernel (generic function with 1 method)

Now we *launch* the kernel on the GPU using the `@cuda` macro. We can specify how many threads and blocks to launch with at this time. **Keep in mind that CUDA kernels cannot see arrays in CPU memory, only GPU arrays.**

In [30]:
cu_a = CUDA.rand(Float32, 512)

@cuda threads=64 blocks=div(length(cu_a), 64) my_first_cuda_kernel(cu_a)

CUDACore.HostKernel for my_first_cuda_kernel(CuDeviceVector{Float32, 1})

Hi! My thread index is 1 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 2 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 3 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 4 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 5 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 6 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 7 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 8 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 9 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 10 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 11 and my block index is 5. The length of my argument is 512.
Hi! My thread index is 12 and my block index is 5. The length of my argume

However, **stack-allocated objects** like scalar integers, floats, or *tuples* (and thus *StaticArrays*) **can** be passed as arguments to CUDA kernels. 😈

In [ ]:
using StaticArrays

static_a = SVector{16, ComplexF64}(rand(ComplexF64) for si in 1:16)
@cuda threads=16 my_first_cuda_kernel(static_a)

CUDACore.HostKernel for my_first_cuda_kernel(SVector{16, ComplexF64})

### Specifying type constraints in CUDA kernel arguments

From the point of view of a CUDA kernel, a `CuArray{T}` is a `CuDeviceArray{T}`. This is a new type that implements GPU-compatible operations, and the conversion is handled automatically for you. You can use multiple dispatch to define different kernels for different element types or array dimension and call the appropriate kernel.

In [ ]:
function my_first_cuda_kernel(a::CuDeviceArray{Float32})
    t_x = threadIdx().x
    b_x = blockIdx().x
    @cuprintln("Hi! My thread index is $t_x and my block index is $b_x. Wowow Float32 so speedy.")
    return
end

function my_first_cuda_kernel(a::CuDeviceArray{Float16})
    t_x = threadIdx().x
    b_x = blockIdx().x
    @cuprintln("Hi! My thread index is $t_x and my block index is $b_x. Float16 so slender.")
    return
end

my_first_cuda_kernel (generic function with 3 methods)

In [ ]:
CUDA.@sync @cuda threads=16 my_first_cuda_kernel(CUDA.rand(Float32, 16))

Hi! My thread index is 1 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 2 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 3 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 4 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 5 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 6 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 7 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 8 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 9 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 10 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 11 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 12 and my block index is 1. The length of my argument is 16.
H

CUDACore.HostKernel for my_first_cuda_kernel(CuDeviceVector{Float32, 1})

In [32]:
CUDA.@sync @cuda threads=16 my_first_cuda_kernel(CUDA.rand(Float16, 16))

Hi! My thread index is 1 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 2 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 3 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 4 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 5 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 6 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 7 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 8 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 9 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 10 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 11 and my block index is 1. The length of my argument is 16.
Hi! My thread index is 12 and my block index is 1. The length of my argument is 16.
H

CUDACore.HostKernel for my_first_cuda_kernel(CuDeviceVector{Float16, 1})

## Avoiding warp divergence

As mentioned above, threads in the same warp executing different instructions forces each thread to execute in serial, dramatically slowing execution. So does this mean we can never use `if` or `else`? Let's do some experiments to find out.

In [31]:
function diverging_kernel(out_arr, arr)
    my_flag = arr[threadIdx().x]
    a = 1.0
    n_iter = my_flag > 0 ? (my_flag + 1)*5 : 20
    for ix in 1:n_iter
        if my_flag > 0
            a *= -2 * sin(rand() * π/4)
        else
            a += sqrt(0.25)
        end
    end
    return
end

diverging_kernel (generic function with 1 method)

Let's try launching and timing this with various values for `arr`.

In [33]:
@cuda threads=64 diverging_kernel(CUDA.zeros(Int, 64));

In [34]:
cu_arr = CuArray(rand(0:1, 64))
CUDA.@profile @cuda threads=64 blocks=50_000 diverging_kernel(cu_arr)

Profiler ran for 52.36 ms, capturing 12 events.

Host-side activity: calling CUDA APIs took 74.15 µs (0.14% of the trace)
┌──────────┬────────────┬───────┬──────────────────┐
│ Time (%) │ Total time │ Calls │ Name             │
├──────────┼────────────┼───────┼──────────────────┤
│    0.13% │   69.62 µs │     1 │ cuLaunchKernelEx │
└──────────┴────────────┴───────┴──────────────────┘

Device-side activity: GPU was busy for 3.29 ms (6.29% of the trace)
┌──────────┬────────────┬───────┬──────────────────────────────────────────────┐
│ Time (%) │ Total time │ Calls │ Name                                         │
├──────────┼────────────┼───────┼──────────────────────────────────────────────┤
│    6.29% │    3.29 ms │     1 │ diverging_kernel(CuDeviceArray<Int64, 1, 1>) │
└──────────┴────────────┴───────┴──────────────────────────────────────────────┘


In [35]:
cu_arr = CuArray(rand(0:1, 67))
CUDA.@profile @cuda threads=64 blocks=50_000 diverging_kernel(cu_arr)

Profiler ran for 19.61 ms, capturing 12 events.

Host-side activity: calling CUDA APIs took 55.55 µs (0.28% of the trace)
┌──────────┬────────────┬───────┬──────────────────┐
│ Time (%) │ Total time │ Calls │ Name             │
├──────────┼────────────┼───────┼──────────────────┤
│    0.27% │   52.93 µs │     1 │ cuLaunchKernelEx │
└──────────┴────────────┴───────┴──────────────────┘

Device-side activity: GPU was busy for 3.29 ms (16.79% of the trace)
┌──────────┬────────────┬───────┬──────────────────────────────────────────────┐
│ Time (%) │ Total time │ Calls │ Name                                         │
├──────────┼────────────┼───────┼──────────────────────────────────────────────┤
│   16.79% │    3.29 ms │     1 │ diverging_kernel(CuDeviceArray<Int64, 1, 1>) │
└──────────┴────────────┴───────┴──────────────────────────────────────────────┘


In [36]:
cu_arr = CuArray(vcat(zeros(Int, 32), ones(Int, 32)))
println(typeof(cu_arr))
CUDA.@profile @cuda threads=64 blocks=50_000 diverging_kernel(cu_arr)

CuArray{Int64, 1, CUDACore.DeviceMemory}


Profiler ran for 17.24 ms, capturing 12 events.

Host-side activity: calling CUDA APIs took 43.63 µs (0.25% of the trace)
┌──────────┬────────────┬───────┬──────────────────┐
│ Time (%) │ Total time │ Calls │ Name             │
├──────────┼────────────┼───────┼──────────────────┤
│    0.24% │   40.77 µs │     1 │ cuLaunchKernelEx │
└──────────┴────────────┴───────┴──────────────────┘

Device-side activity: GPU was busy for 1.65 ms (9.59% of the trace)
┌──────────┬────────────┬───────┬──────────────────────────────────────────────┐
│ Time (%) │ Total time │ Calls │ Name                                         │
├──────────┼────────────┼───────┼──────────────────────────────────────────────┤
│    9.59% │    1.65 ms │     1 │ diverging_kernel(CuDeviceArray<Int64, 1, 1>) │
└──────────┴────────────┴───────┴──────────────────────────────────────────────┘


In [ ]:
cu_arr = CUDA.ones(Int, 64)
CUDA.@profile @cuda threads=64 blocks=50_000 diverging_kernel(cu_arr)

Profiler ran for 17.99 ms, capturing 12 events.

Host-side activity: calling CUDA APIs took 32.42 µs (0.18% of the trace)
┌──────────┬────────────┬───────┬──────────────────┐
│ Time (%) │ Total time │ Calls │ Name             │
├──────────┼────────────┼───────┼──────────────────┤
│    0.17% │   30.04 µs │     1 │ cuLaunchKernelEx │
└──────────┴────────────┴───────┴──────────────────┘

Device-side activity: GPU was busy for 3.29 ms (18.30% of the trace)
┌──────────┬────────────┬───────┬──────────────────────────────────────────────┐
│ Time (%) │ Total time │ Calls │ Name                                         │
├──────────┼────────────┼───────┼──────────────────────────────────────────────┤
│   18.30% │    3.29 ms │     1 │ diverging_kernel(CuDeviceArray<Int64, 1, 1>) │
└──────────┴────────────┴───────┴──────────────────────────────────────────────┘


In [ ]:
cu_arr = CuArray(rand(-1:3, 64))
CUDA.@profile @cuda threads=64 blocks=50_000 diverging_kernel(cu_arr)

Profiler ran for 21.65 ms, capturing 12 events.

Host-side activity: calling CUDA APIs took 32.19 µs (0.15% of the trace)
┌──────────┬────────────┬───────┬──────────────────┐
│ Time (%) │ Total time │ Calls │ Name             │
├──────────┼────────────┼───────┼──────────────────┤
│    0.14% │   30.04 µs │     1 │ cuLaunchKernelEx │
└──────────┴────────────┴───────┴──────────────────┘

Device-side activity: GPU was busy for 6.58 ms (30.38% of the trace)
┌──────────┬────────────┬───────┬──────────────────────────────────────────────┐
│ Time (%) │ Total time │ Calls │ Name                                         │
├──────────┼────────────┼───────┼──────────────────────────────────────────────┤
│   30.38% │    6.58 ms │     1 │ diverging_kernel(CuDeviceArray<Int64, 1, 1>) │
└──────────┴────────────┴───────┴──────────────────────────────────────────────┘


## Working with memory inside kernels

When a warp executes, it has access to 3 types of memory:
- Global memory, or the on-device GPU RAM. This is the largest and slowest memory.
- Local memory, which holds locally defined variables for each thread.
- Shared memory, which is shared (wow, really?) among all the threads in the warp. This memory allows threads **to communicate with one another**.

Shared memory is physically located next to the cores executing the code and is fastest. To allocate shared memory, we can either define a `CuStaticSharedArray(T, N)`, with `N` elements of type `T` (`N` must be known at compile time) or a `CuDynamicSharedArray(T, N)`, where `N` isn't known at compile-time but will be passed to the kernel at *launch* time. **Note** that when you're using shared memory, you should call `sync_threads()` after any updates to it to make sure all threads in the warp can "see" the update (avoiding a data race).

In [ ]:
function reverse_kernel(a::CuDeviceArray{T}) where T
    i = threadIdx().x
    b = CuDynamicSharedArray(T, length(a))
    b[length(a)-i+1] = a[i]
    sync_threads()
    a[i] = b[i]
    return
end

cu_a = CuArray([1, 2, 3, 4, 5, 6, 7, 8])

@cuda threads=length(cu_a) shmem=sizeof(cu_a) reverse_kernel(cu_a)

CUDACore.HostKernel for reverse_kernel(CuDeviceVector{Int64, 1})

The `shmem` argument should be the *total length* of shared memory needed across *all threads*. Many advanced GPU algorithms use shared memory for their effectiveness, so it's worth being aware of. The above example was pretty small -- do you think this would work if `a` were very large? Would we have to modify our kernel?

### Memory access patterns

As we discussed yesterday, the order in which we access array elements can have big performance impacts. In general, accessing adjacent memory locations in GPU memory is **much** faster than accessing locations that are widely separated. This is because to read a single location, the thread(s) have to load a whole "word" of memory at a time, so two memory locations that are part of the same word can be read cheaply once the word is loaded. This is also sometimes called "coalesced" memory access. It's most efficient to align warp-reads with 32-byte boundary memory regions -- something that using *strides* can often help with! How much of an impact can this have? For a matrix-matrix multiplication, NVIDIA ran tests on a V100 GPU. The observed memory bandwith was:

- Naive implementation: `12.8 GB/s`
- With shared memory to coalesce reads from global memory: `140.2 GB/s`
- Further optimizations from reduce bank conflicts: `199.4 GB/s`

Data taken from the [CUDA C++ programming guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#shared-memory). So, if your application involves a lot of reading and writing to memory, it's really worth figuring out how to use shared memory effectively. This is also something the profiler can help with.

## Benchmarking and profiling

`CUDA.jl` comes with a great integration with the NVIDIA CUDA profilers. This makes it easier to profile our kernels and find and fix performance problems. We can use the `CUDA.@profile` macro to see where we're spending our time. Let's revisit our threading example from above.

In [ ]:
n_arrs = 100
CUDA.@profile begin
    cpu_arrays = [rand(Float64, 1024, 1024) for ix_arr in 1:n_arrs]
    results = Vector{Float64}(undef, n_arrs)
    @sync begin
        for ix_arr in 1:n_arrs
            Threads.@spawn begin
                results[ix_arr] = mapreduce(sin, *, CuArray(cpu_arrays[ix_arr]))
            end
        end
    end
end

Profiler ran for 1.11 s, capturing 23806 events.

Host-side activity: calling CUDA APIs took 215.35 ms (19.34% of the trace)
┌──────────┬────────────┬───────┬───────────────────────────────────────┬─────────────────────────┐
│ Time (%) │ Total time │ Calls │ Time distribution                     │ Name                    │
├──────────┼────────────┼───────┼───────────────────────────────────────┼─────────────────────────┤
│   20.84% │   232.1 ms │   100 │   2.32 ms ± 0.66   (  1.86 ‥ 4.14)    │ cuMemcpyHtoDAsync       │
│    8.77% │    97.7 ms │   100 │ 976.98 µs ± 892.05 (  3.58 ‥ 2199.17) │ cuStreamCreate          │
│    2.27% │    25.3 ms │   300 │  84.32 µs ± 145.62 (  3.34 ‥ 947.0)   │ cuMemAllocFromPoolAsync │
│    0.33% │    3.66 ms │   200 │  18.28 µs ± 12.26  (  8.34 ‥ 146.39)  │ cuLaunchKernelEx        │
│    0.30% │    3.38 ms │   100 │  33.78 µs ± 17.62  ( 16.69 ‥ 154.73)  │ cuMemcpyDtoHAsync       │
│    0.11% │    1.19 ms │   200 │   5.97 µs ± 8.53   (  2.62 ‥ 118.97)  │ c

We can also generate a trace to provide to NVIDIA's profile visualizer, which can give us lots of detailed information and tips about what to fix. We can either launch Julia [inside the `nsys` profiler](https://cuda.juliagpu.org/dev/development/profiling/#NVIDIA-Nsight-Systems) or generate the file by setting the `external=true` argument:

In [ ]:
n_arrs = 100
CUDA.@profile external=true begin
    cpu_arrays = [rand(Float64, 1024, 1024) for ix_arr in 1:n_arrs]
    results = Vector{Float64}(undef, n_arrs)
    @sync begin
        for ix_arr in 1:n_arrs
            Threads.@spawn begin
                results[ix_arr] = mapreduce(sin, *, CuArray(cpu_arrays[ix_arr]))
            end
        end
    end
end

You can then open the generated report file in NSight Systems and see a trace of what the GPU was doing when, how much load it was under, and use this information to improve your code (e.g. by overlapping kernel launches and memory copies). You can also use [NVIDIA NSight Compute](https://cuda.juliagpu.org/dev/development/profiling/#NVIDIA-Nsight-Compute) to do very detailed analysis on a single kernel and pinpoint things to fix.

**In GPU programming, you absolutely need to benchmark your kernels and use the profiler to fix any issues.**

## Restrictions when writing kernels

*Not* all Julia functions can be used on GPU. The GPU supports different instructions and has a different compiler pipeline than we are used to. There are many operations we are used to using without thinking which aren't supported. Let's meet some of them:


### Some more general tips and tricks

- If your GPU is new enough to have tensor cores, you should use them if at all possible.
- 32 bit is generally faster than 64 bit (but we need 64 most of the time!).
- Sometimes you come across an LLVM bug 🫠. Sometimes multiple times a month!
- On newer NVIDIA GPUs, 64 bit support is provided via [emulation](https://docs.nvidia.com/cuda/cublas/#floating-point-emulation) from lower-precision floats. You should read the NVIDIA documentation to understand more about this.
- `CUDA.jl` is now a "thin layer" on top of more independent subsidiary packages -- this means you can import **only the ones you need**  (e.g. `cuBLAS` and `cuSOLVER`, but leave `cuFFT` out) to reduce load times.
- Many array methods do **not** yet have an equivalent GPU kernel written in the JuliaGPU ecosystem. Please **do** open an issue if you encounter this!
- Type stability in GPU code is extremely important -- type unstable code will not compile, and figuring out where the problem is can be nontrivial.

## A new tool for CUDA programming: [`cuTile.jl`](https://github.com/JuliaGPU/cuTile.jl)

NVIDIA has released a new library for *tile* based GPU programming. Rather than work on the basis of individual warps and threads, `cuTile` allows us to work on larger tile objects. This is a paradigm that's quite natural for a lot of applications like matrix multiplication, and can lead to substantial speedups as the compiler is able to do additional optimizations. We have an (experimental) Julia wrapper for this software, with performance that is pretty competitive:

| Kernel | Size | Julia | Python | Status |
|--------|------|-------|--------|--------|
| Vector Addition | 2^27 f32 | 844 GB/s | 845 GB/s | OK (=) |
| Matrix Transpose | 8192² f32 | 812 GB/s | 814 GB/s | OK (=) |
| Layer Norm fwd | 4096² f32 | 986 GB/s | 716 GB/s | +38% |
| Layer Norm bwd | 4096² f32 | 246 GB/s | 251 GB/s | OK (-2%) |
| Matrix Multiplication | 4096³ f32 | 47.4 TFLOPS | 43.5 TFLOPS | +9% |
| Batch Matrix Multiply | 1024×512×2048 ×8 f32 | 34.2 TFLOPS | 30.9 TFLOPS | +11% |
| FFT (3-stage Cooley-Tukey) | 4096-pt ×256 c64 | 209 μs | 204 μs | OK (-2%) |
| Mixture of Experts | 256tok 1024h 32e 2048i f16 | 27.7 TFLOPS | 20.3 TFLOPS | +36% |
| Attention (FMHA) | 8×16×1024² ×64 f16 causal | 102.7 TFLOPS | 63.3 TFLOPS | +62% |
| Softmax (TMA) | 4096² f32 | 838 GB/s | 843 GB/s | OK (-1%) |
| Softmax (Chunked) | 4096² f32 | 1672 GB/s | 1636 GB/s | OK (+2%) |

## Some advantages of using Julia for GPU programming

- Easy to write kernels that are human-readable
- Can embed assembly in kernels to squeeze out more performance if necessary
- Good wrappers for most vendor libraries, including higher level APIs from `LinearAlgebra` and co.
- Seamless integration of CUDA streams and `Task`s to make scheduling multiple kernels ergonomic
- Good support for CUDA-aware MPI through `MPI.jl`
- Surprisingly easy to achieve good performance with a small amount of Julia code
- Relatively easy to implement stuff out of the sandbox that CUDA convenience libraries like `cuTile` provide
- Julia's package extension system makes providing optional GPU acceleration much more user-friendly
- A **lot** of Julia code "just works" on CUDA
- Multiple dispatch to kernels is very helpful and cuts down on code bloat

## Some issues remain
- Concrete eval doesn't work particularly well -- Julia's compiler can often evaluate `sin(1)` at compile time, but `CUDA.jl` wants to call the underlying NVIDIA `sin` assembly operation at runtime
- Not everything is wrapped (please open an issue!)
- We do wrap many low level primitives (such as for warp synchronization) but getting within 1% of C++ performance can be hard

## More resources

A lot of the tutorials and resources for CUDA and GPU programming in general are written in C or C++. Many of the concepts apply to writing performant kernels in Julia or any other language. Additionally, some older references don't account for new device features (like tensor cores) that NVIDIA has introduced over the years. The best technique is to implement something and profile it!

- [CUDA C Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide)
- [JuliaCon 2021 GPU workshop](https://github.com/maleadt/juliacon21-gpu_workshop) -- 3 hours worth of tutorial content
- [`CUDA.jl` documentation](https://cuda.juliagpu.org/dev/)
- [`KernelAbstractions.jl`](https://juliagpu.github.io/KernelAbstractions.jl/dev/) -- write once, run on any GPU
- [`DaggerGPU.jl`](https://github.com/JuliaGPU/DaggerGPU.jl) -- graph-based scheduler that can take GPU resources into account
- [Hands on with Julia for HPC on GPUs and CPUs](https://www.youtube.com/watch?v=RNmSqbG2MUc)
- [Differentiable modeling on GPUs workshop](https://github.com/PTsolvers/gpu-workshop-JuliaCon23)
- [`#gpu` channel on Julia slack](https://julialang.org/slack/)
- [GPU category on Julia discourse](https://discourse.julialang.org/c/domain/gpu/11)
- [JuliaGPU Zoom office hours](https://julialang.org/community/), scroll down for the calendar invite

### Deep dive into GEMM kernel optimization

This pair of recent blog posts [(A)](https://jianyuh.github.io/gemm/optimization/hopper/2024/12/29/h100_gemm.html) and [(B)](https://cudaforfun.substack.com/p/outperforming-cublas-on-h100-a-worklog) discusses optimizing the GEMM kernel for H100 CUDA devices. It includes a deep dive into **multiple** steps of optimization, which we can quickly summarize here:

- Take advantage of the H100's tensor cores, which can compute matrix-matrix multiplication in a single instruction for small matrix sizes
- Implement **tiling** of outputs, and use larger output tiles which can help with memory access
- Hide *load* latency by interleaving tensor core ops and memory ops, which can be done with warp specialization
- Use special warpgroups to work with the output tile (prevents register spilling)
- Hide *store* latency by overlapping write operations for one thread block with load operations for another
- Force nearby tiles to be scheduled at the same time to use the L2 cache
- Use a device assembly instruction to implement a slightly faster barrier
- Group multiple thread blocks together into clusters to allow the kernel to load the same tile to multiple SMs
- A bunch of micro optimizations
- Using asynchronous stores to move data from the SM registers back into global memory
- Implement careful scheduling to improve L2 cache hit rate

Even with all this work, the optimized kernel is only **7%** faster than `CUBLAS` in the best case, while `CUBLAS` is **generally** quite performant. Partly this is because NVIDIA (and other vendors) employs an army of very smart people to write performant kernels. There are two main lessons from this:

1. Writing state of the art kernels is really hard
2. You should try to use the vendor library unless you have a good reason not to

## Using CUDA and other GPU libraries for QIS

CUDA provides several libraries besides `CUBLAS` which are useful for us as quantum information scientists:

- `CUTENSOR`, for tensor operations like permutation, contraction (Einstein summation), addition, etc.
- `CUSTATEVEC`, a GPU-accelerated library for state vector simulation of quantum circuits
- `CUDENSITYMAT`, a GPU-accelerated library for density matrix simulation of quantum circuits
- `CUTENSORNET`, a GPU-accelerated library for tensor network simulation of quantum circuits
- `CUPAULIPROP`, a GPU-accelerated library for Pauli propagation simulation
- `CUSTABILIZER`, a GPU-accelerated library for simulation of stabilizers

Many higher level packages provide these as backend simulation options. For example, `ITensors.jl` and the `TensorKit.jl` ecosystems both offer `CUTENSOR` accelerated backends.

Additionally, AMD offers their own `CUTENSOR` equivalent with an extremely similar API which is not yet wrapped by `AMDGPU.jl`.

[`Yao.jl`](https://github.com/QuantumBFS/Yao.jl), a circuit simulation library, also has an extension for CUDA acceleration for simulations.

In general, the higher level Julia libraries provide many convenience functions that the lower level CUDA libraries don't, and are portable to other HPC systems, so often the easier lift is to start with the high level package and file issues with `CUDA.jl` or `AMDGPU.jl` as needed to let maintainers (me & friends) fix things or add functionality.

## More about `KernelAbstractions.jl` and `AcceleratedKernels.jl`

As we said above, CUDA is probably the most mature GPU programming framework around, but AMD has put a lot of effort to improving their ROCm tooling for their GPUs, and Apple (Metal) and Intel (OneAPI) also offer GPU programming toolkits. Happily, Julia has a high level framework for targeting any of these backends, allowing you to write portable code that can run on various GPUs. This is particularly helpful if you have allocations at a variety of supercomputing centers, which tend to buy large sets of GPUs from either NVIDIA or AMD, and at which AMD is quite common.

`AcceleratedKernels.jl` works with `KernelAbstractions.jl` to provide a truly portable platform for writing "accelerated" code, **but**, as the authors state, "The API is starting to stabilise". So, many features and libraries (like sparse support) you may expect from `CUDA.jl` or `AMDGPU.jl` may not be supported yet (PRs welcome!). We can take a look at some sample kernels, again intended to run on NVIDIA hardware since that's what Google CoLab provides.

In [ ]:
Pkg.add("AcceleratedKernels"); Pkg.add("KernelAbstractions")
using AcceleratedKernels, KernelAbstractions

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [ ]:
m = CUDA.rand(Float32, 10, 100_000); # LARGE

In [ ]:
f(x) = x * x
mrowsumsq = AcceleratedKernels.mapreduce(f, +, m; init=zero(eltype(m)), dims=1)

1×100000 CuArray{Float32, 2, CUDACore.DeviceMemory}:
 2.67462  3.72567  3.35629  2.89211  …  4.93701  3.71169  2.86202  3.19164

`AcceleratedKernels.jl` provides many "basic" array operations like `map`, `mapreduce`, `sort` -- you can do a lot of powerful things with these! For writing your own kernels, you can also do some light piracy and work to understand some of how these kernels are implemented under the hood:

In [ ]:
@kernel inbounds=true cpu=false unsafe_indices=true function _mapreduce_block!(@Const(src), dst, f, op, neutral)

    @uniform N = @groupsize()[1]
    sdata = @localmem eltype(dst) (N,)

    len = length(src)

    # NOTE: for many index calculations in this library, computation using zero-indexing leads to
    # fewer operations (also code is transpiled to CUDA / ROCm / oneAPI / Metal code which do zero
    # indexing). Internal calculations will be done using zero indexing except when actually
    # accessing memory. As with C, the lower bound is inclusive, the upper bound exclusive.

    # Group (block) and local (thread) indices
    iblock = @index(Group, Linear) - 0x1
    ithread = @index(Local, Linear) - 0x1

    i = ithread + iblock * (N * 0x2)
    if i >= len
        sdata[ithread + 0x1] = neutral
    elseif i + N >= len
        sdata[ithread + 0x1] = f(src[i + 0x1])
    else
        sdata[ithread + 0x1] = op(f(src[i + 0x1]), f(src[i + N + 0x1]))
    end

    @synchronize()

    if N >= 512u16
        if ithread < 256u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 256u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 256u16
        if ithread < 128u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 128u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 128u16
        if ithread < 64u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 64u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 64u16
        if ithread < 32u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 32u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 32u16
        if ithread < 16u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 16u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 16u16
        if ithread < 8u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 8u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 8u16
        if ithread < 4u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 4u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 4u16
        if ithread < 2u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 2u16 + 0x1])
        end
        @synchronize()
    end
    if N >= 2u16
        if ithread < 1u16
            sdata[ithread + 0x1] = op(sdata[ithread + 0x1], sdata[ithread + 0x1 + 0x1])
        end
        @synchronize()
    end

    # Code below would work on NVidia GPUs with warp size of 32, but create race conditions and
    # return incorrect results on Intel Graphics. It would be useful to have a way to statically
    # query the warp size at compile time
    #
    # if ithread < 32
    #     N >= 64 && (sdata[ithread + 1] = op(sdata[ithread + 1], sdata[ithread + 32 + 1]))
    #     N >= 32 && (sdata[ithread + 1] = op(sdata[ithread + 1], sdata[ithread + 16 + 1]))
    #     N >= 16 && (sdata[ithread + 1] = op(sdata[ithread + 1], sdata[ithread + 8 + 1]))
    #     N >= 8 && (sdata[ithread + 1] = op(sdata[ithread + 1], sdata[ithread + 4 + 1]))
    #     N >= 4 && (sdata[ithread + 1] = op(sdata[ithread + 1], sdata[ithread + 2 + 1]))
    #     N >= 2 && (sdata[ithread + 1] = op(sdata[ithread + 1], sdata[ithread + 1 + 1]))
    # end

    if ithread == 0x0
        dst[iblock + 0x1] = sdata[0x1]
    end
end

_mapreduce_block! (generic function with 4 methods)

Wow! That is complicated and a big pain. However, this shows the amount of work that can go into writing a "top line" GPU kernel.

### Tuning

To get good performance out of a particular GPU for a particular use case, you may need to do (a **lot** of) "tuning". This involves benchmarking various usage scenarios -- more or less square matrix, "hot dog shaped" (technical term) matrix, triangular matrix, etc. -- on the real hardware and saving the results, which can then be used in a large decision tree to pick which kernel to launch. The tuning process can take many days or even weeks, and the results can be very large -- this is why some library shared object files are humongous (100s of MB). To perform tuning you will probably need a large amount of compute, so it is hard to do if you yourself aren't a GPU vendor or a supercomputing center.

## One more interesting package: `GemmKernels.jl`

A very common operation we'd like to perform on GPU is `GEMM` - general matrix multiplication. This operation is of extreme importance to ML/AI applications and GPU vendors and AI startups spend a lot of money to find people who can write good kernels for this. JuliaGPU has a package attemping to write some good open source GEMM kernels at `GemmKernels.jl` -- but the tuning process takes up to a week 🤯.

## But I don't HAVE an NVIDIA GPU -- what are my options, then?

In fact, many supercomputing centers are AMD only (for a few reasons), and if you have a newer MacBook, you may have integrated Apple graphics via a chip that supports Metal. One big advantage of the JuliaGPU pocket universe is that we have a strong suite of common tools that work with almost any GPU backend, such as:

- [`GPUArrays.jl`](https://github.com/juliagpu/gpuarrays.jl), which offers a backend-agnostic array API and kernels for GPU-based array operations, like `mapreduce`, `matmatmul`, `findmin/findmax`, etc. Now with support for sparse arrays!
- [`GPUCompiler.jl`](https://github.com/JuliaGPU/GPUCompiler.jl), which takes our high level Julia code and compiles it directly to various GPUs through LLVM.
- [`KernelAbstractions.jl`](https://github.com/juliagpu/kernelabstractions.jl), lets us write one kernel and run it on any GPU backend (thanks to `GPUCompiler.jl`), very helpful for writing **one** code that can run on many different types of hardware.
- [`Adapt.jl`](https://github.com/juliagpu/adapt.jl), allows us to define logic for moving memory around from device to device (CPU <-> GPU, for example).

Then we have device-specific packages:

- [`CUDA.jl`](https://github.com/juliagpu/cuda.jl), which we already saw.
- [`AMDGPU.jl`](https://github.com/juliagpu/amdgpu.jl), which targets AMD devices and their software library, ROCm.
- [`Metal.jl`](https://github.com/juliagpu/metal.jl), which targets Apple devices.

What makes the code we have written *portable*?

1. Does it **run** on the different platforms I need?
2. Are the results **correct** each place I'm running it? Can I develop on AMD hardware and run later on NVIDIA without too much debugging?
3. Is the **performance** comparable?
4. Can I use **hardware specific optimizations** without having to duplicate all my functions?

## Let's write some `KernelAbstractions.jl` code!

One really helpful thing about KA is that we can target the CPU as a backend, so even if you don't have a GPU, you can still develop. In `KernelAbstractions.jl`, we use a macro to create a kernel:

```julia
@kernel my_kernel(args...)
    # some stuff
end
```

🚨 A `KernelAbstractions` `@kernel` **must not** use the `return` keyword! 🚨

The workflow for `KA` ideally goes like:
1. Write the kernel using the `@kernel` macro.
2. *Instantiate* the kernel for a *specific* backend `kernel = my_kernel(backend)`, like `CUDABackend()`.
3. Allocate some memory with `KernelAbstractions.allocate(backend, ...)`.
4. *Launch* the kernel with a specification of the grid to run over.
5. (Eventually) *synchronize* the backend to get the results of the kernel execution.

Let's examine this by comparing with a CPU function we understand.

In [ ]:
N = 1024^2

1048576

In [ ]:
function vecadd!(a, b, c)
	for i in eachindex(c)
		c[i] = a[i] + b[i]
	end
end

a = rand(N)
b = rand(N)
c = similar(a)

vecadd!(a, b, c)

How can we write this as a `KernelAbstractions` kernel?

In [ ]:
using KernelAbstractions: @kernel, @index

@kernel function vecadd_kernel(a, b, c)
	i = @index(Global)
	c[i] = a[i] + b[i]
end

backend = CUDABackend()
a = KernelAbstractions.allocate(backend, Float32, N)
b = KernelAbstractions.allocate(backend, Float32, N)
c = similar(a)

vadd_kernel = vecadd_kernel(backend)
vadd_kernel(a, b, c; ndrange=size(c))

Now we can benchmark this. 🚨 We **must** synchronize during benchmarking, otherwise we're only measuring the time to launch the kernel! 🚨

In [ ]:
Pkg.add("Chairmarks")
using Chairmarks
@b begin
	vadd_kernel(a, b, c; ndrange=size(c))
	KernelAbstractions.synchronize(backend)
end

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


79.479 μs (79 allocs: 1.719 KiB)

### Some things to consider about this kernel
- What is `@index(Global)`? How might it differ from `@index(Linear)`?
- Although we called this `vecadd`, what happens if you create *matrices* for `c`, `a`, `b`? Does the kernel still work?

### Let's try something a little more complex: a `sum` implementation

Suppose you want to sum up the values of an array `arr` after applying some function `f` to each element. In effect, we want to write

```julia
sum(f, a) = sum([f(a_) for a_ in a])
```

Can we write a kernel to do this?

In [ ]:
@kernel function my_first_sum(val, f, data)
	I = @index(Global)
	x = data[I]
  # since we cannot return a scalar, we accumulate into a length-1 device vector
	val[1] += f(x)
end

data = KernelAbstractions.ones(backend, Float64, 1024^2);
sum(identity, data)

1.048576e6

In [ ]:
let
	x = KernelAbstractions.zeros(backend, eltype(data), 1)
	my_first_sum(backend)(x, identity, data, ndrange=length(data))
	Array(x)[1]
end

26.0

Hmmm... this doesn't seem right. What could be going wrong? What happens if you run the kernel again?

The cause of the error is a data race. We can fix this by using the package `Atomix.jl` which allows us to implement atomic operations on the GPU in a device-agnostic way.

In [ ]:
Pkg.add("Atomix")
using Atomix
using Atomix: @atomic, @atomicswap, @atomicreplace

@kernel function my_atomic_sum(val, f, data)
	I = @index(Global)
	x = f(data[I])
	@atomic val[1] += x
end

x = KernelAbstractions.zeros(backend, eltype(data), 1)
my_atomic_sum(backend)(x, identity, data, ndrange=length(data))
Array(x)[1]

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


1.048576e6

OK, that looks more or less correct. Let's now examine the performance with some benchmarking and profiling.

In [ ]:
x = KernelAbstractions.zeros(backend, eltype(data), 1)
@b begin
	my_atomic_sum($backend)($x, identity, data, ndrange=length(data))
	KernelAbstractions.synchronize($backend)
end

3.889 ms (308 allocs: 5.359 KiB)

In [ ]:
@b sum(identity, data)

101.827 μs (74 allocs: 2.094 KiB)

Why might this be?


We are spending a LOT of time in atomic operations. The reason for this is we are doing very little work before each atomic, so there is high contention for the lock. This is something we can resolve with shared memory among the working threads.

In [ ]:
using KernelAbstractions: @uniform, @groupsize, @localmem, @synchronize
@kernel function my_shmem_sum(val, f, data::AbstractArray{T}) where T
	I = @index(Global)
	i = @index(Local)
	N = @uniform prod(@groupsize())
	tile = @localmem T (N,)
	tile[i] = f(data[I])
	@synchronize
	if i == 1
		acc = tile[1]
		for i in 2:N
			acc += tile[i]
		end
		@atomic val[1] += acc
	end
end

my_shmem_sum (generic function with 4 methods)

Let's check again that we get the correct result:

In [ ]:
x = KernelAbstractions.zeros(backend, eltype(data), 1)
my_shmem_sum(backend, 64)(x, identity, data, ndrange=length(data))
Array(x)[1]

1.048576e6

And let's do some more benchmarking:

In [ ]:
x = KernelAbstractions.zeros(backend, eltype(data), 1)
@b begin
    my_shmem_sum($backend, 64)($x, identity, data, ndrange=length(data))
    KernelAbstractions.synchronize($backend)
end

925.678 μs (296 allocs: 5.188 KiB)

OK, this looks substantially better! There are even *more* optimizations we could try, such as:
- Warp-level collectives on CUDA and AMD
- Distributing the work more evenly among the threads in the reduction step
- Giving each group of threads more work to do

## Some exploration ideas

- Try updating the functions we wrote yesterday for the state-vector simulation to use `KernelAbstractions.jl`
- Can you write a GPU accelerated Clifford simulator?
- Can you extend our `sum(f, x)` function to sparse arrays?
- How does the performance of the kernel change with varying element types of `data`? Can you use types like `Int8`?
- Can you extend `my_shmem_sum` to accept multiple arrays and a multi-argument `f`?
- Suppose you want to run `my_shmem_sum` on the rows or columns of a matrix. Which do you think would be more efficient? Can you implement this and test it?